In [2]:
import os.path
import cv2
import torch
import albumentations as A
import torch.nn as nn

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(
        f"GPU count: {torch.cuda.device_count()}"
        f", CUDA version: {torch.version.cuda}"
        f", cuDNN version: {torch.backends.cudnn.version()}"
    )
elif torch.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Device: ', device)

Device:  cpu


In [ ]:
class CityScapsDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, mask_dir, transforms=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transforms = transforms

        self.images = []

        for city in os.listdir(img_dir):
            city_path = os.path.join(img_dir, city)
            for image in os.listdir(city_path):
                self.images.append(os.path.join(city, image))

    def __len__(self):
        return sum([len(files) for m, r, files in os.walk(self.img_dir)])

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx].replace("leftImg8bit", "gtFine_labelIds"))

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, 0)

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image , mask = augmented['image'], augmented['mask']

        return image, mask

In [ ]:
transforms_deep_lab_v3 = A.Compose([A.RandomScale(scale_limit = 0.5),
                                    A.RandomCrop(512, 1024),
                                    A.HorizontalFlip(p=0.5),
                                    A.ColorJitter(0.2, 0.2, 0.2, 0.1),
                                    A.Normalize(mean=(0.485, 0.456, 0.406),
                                                std=(0.229, 0.224, 0.225))])

In [ ]:
class SeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dilation=1, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, stride=stride, dilation=dilation, padding=padding, kernel_size=(3, 3), groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


In [ ]:
class SimpleConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0 ):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )


    def forward(self, x):
        return self.block(x)

In [ ]:
class XceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            SeparableConv(in_channels, out_channels, padding=1),
            SeparableConv(out_channels, out_channels, padding=1),
            SeparableConv(out_channels, out_channels, stride, padding)
        )
        self.skip = None
        if stride!=1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )


    def forward(self, x):
        residual = x
        if self.skip is not None:
            residual = self.skip(x)
        out = self.block(x)
        return out + residual

In [ ]:
class XceptionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([
            XceptionBlock(in_channels=64, out_channels=128, stride=2, padding=1),
            XceptionBlock(in_channels=128, out_channels=256, stride=2, padding=1),
            XceptionBlock(in_channels=256, out_channels=728, stride=2, padding=1)
        ])
        self.conv_2d_32 = SimpleConv(in_channels=3, out_channels=32, stride=2, padding=1, kernel_size=(3,3))
        self.conv_2d_64 = SimpleConv(in_channels=32, out_channels=64, kernel_size=(3,3), padding=1)
        self.middle_layers = nn.ModuleList()
        for i in range(16):
            self.middle_layers.append(XceptionBlock(in_channels=728, out_channels=728, padding=1))
        self.xception_block_1024 = XceptionBlock(in_channels=728, out_channels=1024, padding=1)
        self.sep_conv_1536_1 = SeparableConv(1024, 1536, stride=1, dilation=2, padding=2)
        self.sep_conv_1536_2 = SeparableConv(1536, 1536, stride=1, dilation=2, padding=2)
        self.sep_conv_2048_2 = SeparableConv(1536, 2048, stride=1, dilation=2, padding=2)

    def forward(self, x):
        skip_dec = None
        out = self.conv_2d_32(x)
        out = self.conv_2d_64(out)

        for i, block in enumerate(self.blocks):
            out = block(out)
            if i == 0:
                skip_dec = out
        for layer in self.middle_layers:
            out = layer(out)
        out = self.xception_block_1024(out)
        out = self.sep_conv_1536_1(out)
        out = self.sep_conv_1536_2(out)
        out = self.sep_conv_2048_2(out)

        return out, skip_dec

In [ ]:
class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels, dilations, dropout_aspp):
        super().__init__()
        self.point_conv2d_2048 = SimpleConv(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        self.atrous_convs = nn.ModuleList([
            SeparableConv(in_channels=in_channels, out_channels=out_channels, dilation=rate, padding=rate)
            for rate in dilations
        ])

        self.global_avg_pooling = nn.AdaptiveAvgPool2d(1)
        self.point_conv2d_2048_2 = SimpleConv(in_channels=in_channels, out_channels=out_channels, kernel_size=1)
        self.point_conv2d_1280_3 = SimpleConv(in_channels=1280, out_channels=out_channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout_aspp)

    def forward(self, x):
        out1 = self.point_conv2d_2048(x)
        out2 = [conv(x) for conv in self.atrous_convs]

        out3 = self.global_avg_pooling(x)
        out3 = self.point_conv2d_2048_2(out3)
        out3 = nn.functional.interpolate(out3, size=x.shape[2:], mode='bilinear', align_corners=True)
        out = torch.cat([out1] + out2 + [out3], 1)
        out = self.point_conv2d_1280_3(out)
        out = self.dropout(out)# maybe remove Dropout


        return out


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, num_classes, low_level_channels=128):#in_channels = 256
        super().__init__()
        self.point_conv2d_48 = SimpleConv(in_channels=low_level_channels, out_channels=48, kernel_size=1)
        self.simple_conv2d_256_1 = SimpleConv(in_channels=304, out_channels=in_channels, kernel_size=3, padding=1)
        self.simple_conv2d_256_2 = SimpleConv(in_channels=in_channels, out_channels=in_channels, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.1)
        self.point_conv2d_final = nn.Conv2d(in_channels=in_channels, out_channels=num_classes, kernel_size=1)

    def forward(self, low_features, high_features, input_shape):
        low_out = self.point_conv2d_48(low_features)
        high_out = nn.functional.interpolate(high_features, size=low_features.shape[2:], mode='bilinear', align_corners=True)
        out = torch.cat([low_out, high_out], 1)
        out = self.simple_conv2d_256_1(out)
        out = self.simple_conv2d_256_2(out)
        out = self.dropout(out)
        out = self.point_conv2d_final(out)
        out = nn.functional.interpolate(out, size=input_shape, mode='bilinear', align_corners=True)


        return out



In [ ]:
class DeepLabV3Plus(nn.Module):
    def __init__(self, num_classes, encoder=None):
        super().__init__()
        self.encoder = encoder if encoder is not None else XceptionModel()
        self.aspp = ASPP(in_channels=2048, out_channels=256, dilations=[6, 12, 18], dropout_aspp=0.5)
        self.decoder = DecoderBlock(256, num_classes=num_classes)

    def forward(self, x):
        out, low_features = self.encoder(x)
        high_features = self.aspp(out)
        out = self.decoder(low_features, high_features, x.shape[2:])

        return out

## Training pipeline — pretrained encoder

This part trains the architecture above end-to-end on Cityscapes. Sized deliberately to use most of a **~25 Colab compute unit** budget (see the "compute budget" note further down) rather than a minimal fast run — with that much headroom, it's better spent training longer and getting a genuinely converged model than rushing through a token 5-epoch demo.

Same reasoning as the SAR-ship project: `XceptionModel` above is kept as the from-scratch reference, but training actually uses an **ImageNet-pretrained Xception** (`timm`), or it would need far more compute than even this budget allows (see the from-scratch time estimate in the README).

In [ ]:
!pip install -q timm grad-cam

import time
import json as jsonlib
import shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import ClassVar

import numpy as np
import timm
import matplotlib.pyplot as plt
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Subset


### Label mapping: labelIds → trainIds

Cityscapes' raw `gtFine_labelIds` masks have 34 label IDs (includes things like `license plate`, `bridge`, `rectification border` that nobody trains on). The standard benchmark protocol collapses these into **19 train classes** plus an ignore index (255) for everything else — this is the same mapping `cityscapesScripts` and every published Cityscapes leaderboard entry uses. The original `CityScapsDataset` above loads raw `labelIds` directly with no mapping — training on that as-is would treat 34 raw IDs (many meaningless, heavily imbalanced) as the target classes, which is not the standard/comparable setup.

In [ ]:
NUM_CLASSES = 19
IGNORE_INDEX = 255

# index i = raw labelId, value = trainId (255 = ignore). Standard Cityscapes mapping.
LABELID_TO_TRAINID = np.full(256, IGNORE_INDEX, dtype=np.uint8)
_MAPPING = {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
    23: 10, 24: 11, 25: 12, 26: 13, 27: 14, 28: 15, 31: 16, 32: 17, 33: 18,
}
for labelid, trainid in _MAPPING.items():
    LABELID_TO_TRAINID[labelid] = trainid


### Dataset download

Cityscapes requires a free account at [cityscapes-dataset.com](https://www.cityscapes-dataset.com/) — unlike HRSID there's no anonymous Kaggle-style mirror with matching license terms. `cityscapesscripts`' `csDownload` handles login itself (interactive, `getpass`-protected prompt the first time, with an offer to save credentials to `~/.local/share/cityscapes/credentials.json` for reuse) — it takes **no username/password CLI flags**, so don't try to script those in. Package names below (`gtFine_trainvaltest.zip`, `leftImg8bit_trainvaltest.zip`) are correct as of the dataset's current release; run `!csDownload -l` to list all available packages if either name 404s.

**Drive caching**: the ~11GB download takes 30-45+ minutes, so the cell below caches the raw zips to `/content/drive/MyDrive/CityscapesSegmentation-experiment/dataset_cache/` after a successful extraction. Future runs (new Colab session, runtime restart) copy from Drive instead of re-downloading from cityscapes-dataset.com. It only caches after verifying `leftImg8bit/train` and `gtFine/train` actually extracted — a partial/interrupted download never gets written to the cache, so a bad run can't poison future ones.

**Security note**: if you ever see your password echoed back in a cell's output (e.g. from a malformed shell command), treat that password as compromised and change it immediately — Colab persists cell output in the saved notebook.

In [ ]:
!pip install -q cityscapesscripts

from google.colab import drive

CITYSCAPES_ROOT = Path("./data/cityscapes")
CITYSCAPES_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_CACHE_DIR = Path("/content/drive/MyDrive/CityscapesSegmentation-experiment/dataset_cache")
PACKAGE_NAMES = ["gtFine_trainvaltest.zip", "leftImg8bit_trainvaltest.zip"]


def _extracted_ok() -> bool:
    return (CITYSCAPES_ROOT / "leftImg8bit" / "train").exists() and (CITYSCAPES_ROOT / "gtFine" / "train").exists()


if not _extracted_ok():
    drive.mount("/content/drive")
    DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    cached = all((DRIVE_CACHE_DIR / name).exists() for name in PACKAGE_NAMES)
    if cached:
        print("[drive] found cached zips -- copying locally instead of re-downloading")
        for name in PACKAGE_NAMES:
            shutil.copy2(DRIVE_CACHE_DIR / name, CITYSCAPES_ROOT / name)
    else:
        print("[download] no Drive cache found -- downloading from cityscapes-dataset.com (~11GB total, expect 30-45+ min)")
        # csDownload has no -u/-p flags -- prompts for username/password itself
        !csDownload -d "$CITYSCAPES_ROOT" gtFine_trainvaltest.zip leftImg8bit_trainvaltest.zip

    for name in PACKAGE_NAMES:
        zip_path = CITYSCAPES_ROOT / name
        if zip_path.exists():
            shutil.unpack_archive(str(zip_path), str(CITYSCAPES_ROOT))

    if not _extracted_ok():
        raise RuntimeError(
            "Extraction didn't produce leftImg8bit/train + gtFine/train -- the download was likely "
            "interrupted (runtime disconnect, or you ran the next cell before this one finished). "
            "Delete the partial files under data/cityscapes/ and rerun this cell before continuing."
        )

    if not cached:
        print("[drive] caching zips to Drive for future runs")
        for name in PACKAGE_NAMES:
            shutil.copy2(CITYSCAPES_ROOT / name, DRIVE_CACHE_DIR / name)
else:
    print("[skip] Cityscapes already extracted at", CITYSCAPES_ROOT)


In [ ]:
class PretrainedXceptionEncoder(nn.Module):
    """Drop-in replacement for XceptionModel: same (high_features, low_level_features) output contract the ASPP/DecoderBlock above expect, but backed by ImageNet-pretrained weights via timm."""
    def __init__(self, low_level_channels=128, high_level_channels=2048, model_name="xception"):
        super().__init__()
        # NOTE: if plain "xception" errors on features_only=True in your timm version,
        # switch model_name to "xception41".
        self.backbone = timm.create_model(model_name, pretrained=True, features_only=True)
        feat_channels = self.backbone.feature_info.channels()

        self.low_idx = 1
        self.high_idx = -1

        self.low_proj = nn.Conv2d(feat_channels[self.low_idx], low_level_channels, kernel_size=1)
        self.high_proj = nn.Conv2d(feat_channels[self.high_idx], high_level_channels, kernel_size=1)

    def forward(self, x):
        feats = self.backbone(x)
        low = self.low_proj(feats[self.low_idx])
        high = self.high_proj(feats[self.high_idx])
        return high, low


### Fixed dataset + transforms

Two real bugs fixed relative to the original `CityScapsDataset`/`transforms_deep_lab_v3` cells above (kept as-is for reference, not used for training):
1. **BGR/RGB** — `cv2.imread` returns BGR; never converted, but `Normalize` uses RGB-ordered ImageNet stats.
2. **No tensor conversion** — the original transform pipeline never converts to a `(C,H,W)` tensor via `ToTensorV2`. Albumentations returns `(H,W,C)` numpy arrays, which `DataLoader`'s default collate silently turns into `(N,H,W,C)` tensors — wrong axis order for `nn.Conv2d`, which expects channels first. This would have crashed (or silently misbehaved) the moment training was attempted.

In [ ]:
class CityscapesDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, mask_dir, transforms=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.transforms = transforms
        self.images = []
        for city_dir in sorted(self.img_dir.iterdir()):
            for image_path in sorted(city_dir.glob("*.png")):
                self.images.append(image_path.relative_to(self.img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        rel_path = self.images[idx]
        img_path = self.img_dir / rel_path
        mask_path = self.mask_dir / str(rel_path).replace("leftImg8bit", "gtFine_labelIds")

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), 0)
        mask = LABELID_TO_TRAINID[mask]  # labelId -> trainId (0-18, 255=ignore)

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        return image, mask.long()


train_transforms = A.Compose([
    A.RandomScale(scale_limit=0.5),
    A.RandomCrop(512, 1024),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(0.2, 0.2, 0.2, 0.1),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.PadIfNeeded(min_height=1024, min_width=2048),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

CITYSCAPES_IMG_DIR = CITYSCAPES_ROOT / "leftImg8bit"
CITYSCAPES_MASK_DIR = CITYSCAPES_ROOT / "gtFine"

train_ds = CityscapesDataset(CITYSCAPES_IMG_DIR / "train", CITYSCAPES_MASK_DIR / "train", transforms=train_transforms)
val_ds = CityscapesDataset(CITYSCAPES_IMG_DIR / "val", CITYSCAPES_MASK_DIR / "val", transforms=val_transforms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
# batch_size=1: val runs at full 2048x1024 resolution (unlike the 512x1024 training crops), which
# risks OOM on a T4's 16GB at batch>=2 through this deep encoder+decoder.
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)
print(f"train={len(train_ds)}  val={len(val_ds)}")


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)


@torch.no_grad()
def mean_iou(logits, target, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
    preds = logits.argmax(dim=1)
    valid = target != ignore_index
    ious = []
    for c in range(num_classes):
        pred_c = (preds == c) & valid
        target_c = (target == c) & valid
        union = (pred_c | target_c).sum().item()
        if union == 0:
            continue
        ious.append((pred_c & target_c).sum().item() / union)
    return float(np.mean(ious)) if ious else 0.0


### Checkpoint system

Same pattern as the SAR-ship project's `HRSID_Segmentation.ipynb` (and its VisDrone origin): Colab's `/content` is wiped every fresh runtime, so checkpoints/metrics mirror to Google Drive, with skip-if-already-done resume logic.

In [ ]:
from google.colab import drive

DRIVE_ROOT = Path("/content/drive/MyDrive/CityscapesSegmentation-experiment")


def mount_drive() -> None:
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"[drive] mounted -> {DRIVE_ROOT}")


def save_to_drive(local: Path, relative: str | None = None) -> None:
    relative = relative or local.name
    dst = DRIVE_ROOT / relative
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local, dst)


def load_from_drive(relative: str, local: Path) -> bool:
    src = DRIVE_ROOT / relative
    if not src.exists():
        return False
    local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, local)
    return True


@dataclass
class ExperimentMeta:
    _DRIVE_KEY: ClassVar[str] = "experiment_meta.json"
    histories: dict = field(default_factory=dict)
    metrics: dict = field(default_factory=dict)
    extra: dict = field(default_factory=dict)

    def save(self, local_dir: Path = Path("models")) -> Path:
        local_dir.mkdir(parents=True, exist_ok=True)
        local_path = local_dir / "experiment_meta.json"
        local_path.write_text(jsonlib.dumps(
            {"histories": self.histories, "metrics": self.metrics, "extra": self.extra}, indent=2))
        save_to_drive(local_path, self._DRIVE_KEY)
        return local_path

    @classmethod
    def load(cls, local_dir: Path = Path("models")) -> "ExperimentMeta":
        local_path = local_dir / "experiment_meta.json"
        found = load_from_drive(cls._DRIVE_KEY, local_path)
        if not found and not local_path.exists():
            print("[meta] no saved state found -- starting fresh")
            return cls()
        blob = jsonlib.loads(local_path.read_text())
        return cls(histories=blob["histories"], metrics=blob["metrics"], extra=blob["extra"])


mount_drive()
meta = ExperimentMeta.load()


### Compute budget

Target: **~25 hours of T4 time** (at your observed rate of ~1.1 compute units/hour, that's roughly **27-28 compute units**). Epoch counts below are sized against that, split so the fine-tuned run — the actual deliverable model — gets more of the budget than the frozen baseline:

| Stage | Est. time |
|---|---|
| Setup + Cityscapes download | ~20-30 min |
| Train `cityscapes_frozen` (88 epochs) | ~8.5-9 hours |
| Train `cityscapes_finetuned` (154 epochs) | ~15-15.5 hours |
| Grad-CAM + failure case analysis | ~20 min |
| **Total** | **~24.5-25.5 hours ≈ 27-28 compute units at 1.1/hr** |

Per-epoch time is an **estimate** (~6 min/epoch at batch 4, 512×1024, on a T4) — not measured, since training hasn't actually been run yet. Watch Colab's usage panel after the first few epochs of `cityscapes_frozen`: if the real per-epoch time is higher or lower, adjust `EPOCHS_FROZEN`/`EPOCHS_FINETUNED` below before committing to the full run, rather than discovering the mismatch after burning most of the budget.

In [ ]:
EPOCHS_FROZEN = 88
EPOCHS_FINETUNED = 154


def run_epoch(model, loader, optimizer=None, freeze_bn_in: nn.Module | None = None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    if is_train and freeze_bn_in is not None:
        freeze_bn_in.eval()

    total_loss, total_iou, n_batches = 0.0, 0.0, 0
    with torch.set_grad_enabled(is_train):
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            loss = criterion(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_iou += mean_iou(logits, masks)
            n_batches += 1

    return total_loss / n_batches, total_iou / n_batches


def train_one_variant(run_name: str, freeze_encoder: bool, epochs: int, lr: float = 1e-3,
                       skip_existing: bool = True) -> nn.Module:
    ckpt_path = Path("models") / f"{run_name}.pt"
    model = DeepLabV3Plus(num_classes=NUM_CLASSES, encoder=PretrainedXceptionEncoder()).to(device)

    if skip_existing and run_name in meta.metrics and load_from_drive(f"{run_name}.pt", ckpt_path):
        print(f"[skip] {run_name} -- already trained, loading checkpoint from Drive")
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        return model

    if freeze_encoder:
        for p in model.encoder.backbone.parameters():
            p.requires_grad = False

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    history = {"train": [], "val": []}
    best_val = float("inf")
    best_val_iou = 0.0
    frozen_module = model.encoder.backbone if freeze_encoder else None

    for epoch in range(epochs):
        train_loss, _ = run_epoch(model, train_loader, optimizer, freeze_bn_in=frozen_module)
        val_loss, val_iou = run_epoch(model, val_loader, optimizer=None)
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        print(f"[{run_name}] epoch {epoch+1}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}  val_mIoU={val_iou:.4f}")

        if val_loss < best_val:
            best_val, best_val_iou = val_loss, val_iou
            ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), ckpt_path)
            save_to_drive(ckpt_path, f"{run_name}.pt")

    meta.histories[run_name] = history
    meta.metrics[run_name] = {"val_loss": best_val, "val_mIoU": best_val_iou}
    meta.save()

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return model


model_frozen = train_one_variant("cityscapes_frozen", freeze_encoder=True, epochs=EPOCHS_FROZEN)
model_finetuned = train_one_variant("cityscapes_finetuned", freeze_encoder=False, epochs=EPOCHS_FINETUNED, lr=3e-4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, run_name in zip(axes, ["cityscapes_frozen", "cityscapes_finetuned"]):
    hist = meta.histories[run_name]
    ax.plot(hist["train"], label="train")
    ax.plot(hist["val"], label="val", linestyle="--")
    ax.set_title(run_name)
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

print("Final metrics:")
for run_name in ["cityscapes_frozen", "cityscapes_finetuned"]:
    print(f"  {run_name}: {meta.metrics[run_name]}")


**Write here**: which variant won on val mIoU? With 19 classes and a much bigger compute budget than the SAR project, this comparison should be more decisive than HRSID's — note whether fine-tuning's advantage grows or shrinks relative to the smaller HRSID run.

## Experiment — Grad-CAM

Multi-class version: pick one predicted class per run (default: `car`, trainId 13) and sum the logit map over pixels predicted as that class.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

CITYSCAPES_TRAINID_NAMES = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle",
]


class SemanticSegmentationTarget:
    """Sums a SINGLE class channel's logit over the predicted-that-class region.
    (Unlike the binary HRSID case, model_output has 19 channels here -- indexing
    only `category` avoids summing every other class's logit into the target too.)"""
    def __init__(self, category: int, mask):
        self.category = category
        self.mask = torch.from_numpy(mask).to(device)

    def __call__(self, model_output):
        return (model_output[self.category, :, :] * self.mask).sum()


def run_gradcam(model, image_tensor, image_rgb, target_class_id=13):
    model.eval()
    with torch.no_grad():
        pred = model(image_tensor.unsqueeze(0).to(device)).argmax(dim=1)[0].cpu().numpy()
    class_mask = (pred == target_class_id).astype(np.float32)

    target_layers = [model.aspp.point_conv2d_1280_3.block[0]]
    targets = [SemanticSegmentationTarget(target_class_id, class_mask)]

    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0).to(device), targets=targets)[0]
    return show_cam_on_image(image_rgb, grayscale_cam, use_rgb=True), pred


fig, axes = plt.subplots(3, 3, figsize=(12, 12), constrained_layout=True)
for i in range(3):
    img_t, mask_t = val_ds[i]
    img_rgb = (img_t.permute(1, 2, 0).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]).clip(0, 1)
    overlay, pred = run_gradcam(model_finetuned, img_t, img_rgb, target_class_id=13)  # 13 = car

    axes[i, 0].imshow(img_rgb); axes[i, 0].set_title("image")
    axes[i, 1].imshow(pred, cmap="tab20"); axes[i, 1].set_title("predicted segmentation")
    axes[i, 2].imshow(overlay); axes[i, 2].set_title("Grad-CAM (car)")
    for ax in axes[i]:
        ax.axis("off")
plt.show()


**Write here**: does activation for the `car` class stay on car pixels, or bleed onto visually similar classes (van-shaped trucks, road surface reflections)? Try swapping `target_class_id` to a rare/thin class (e.g. `5` = pole, `17` = motorcycle) and compare — thin classes are where Cityscapes segmentation models typically struggle most, and Grad-CAM on a thin class is a much more informative failure lens than on a large class like `road`.

## Experiment — Failure case analysis

With 19 classes instead of HRSID's 1, "confident but wrong" is richer here — mine for images where mean per-pixel confidence (softmax max-prob) is high but mean IoU across present classes is low.

In [ ]:
@torch.no_grad()
def collect_failure_candidates(model, loader, max_images=200):
    model.eval()
    records = []
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        conf, preds = probs.max(dim=1)

        for i in range(images.shape[0]):
            valid = masks[i] != IGNORE_INDEX
            if valid.sum() == 0:
                continue
            img_iou = mean_iou(logits[i:i+1], masks[i:i+1])
            mean_conf = conf[i][valid].mean().item()
            records.append({
                "image": images[i].cpu(), "mask": masks[i].cpu(), "pred": preds[i].cpu(),
                "iou": img_iou, "confidence": mean_conf,
            })
        if len(records) >= max_images:
            break
    return records


records = collect_failure_candidates(model_finetuned, val_loader)
records.sort(key=lambda r: r["confidence"] - r["iou"], reverse=True)

fig, axes = plt.subplots(4, 3, figsize=(12, 16), constrained_layout=True)
for i, rec in enumerate(records[:4]):
    img_rgb = (rec["image"].permute(1, 2, 0).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]).clip(0, 1)
    axes[i, 0].imshow(img_rgb); axes[i, 0].set_title("image")
    axes[i, 1].imshow(rec["mask"], cmap="tab20"); axes[i, 1].set_title("ground truth")
    axes[i, 2].imshow(rec["pred"], cmap="tab20")
    axes[i, 2].set_title(f"pred (mIoU={rec['iou']:.2f}, conf={rec['confidence']:.2f})")
    for ax in axes[i]:
        ax.axis("off")
plt.show()


**Write here**: which classes dominate the worst failures — thin/rare classes (pole, traffic sign, rider) or large/common ones (road, building) under unusual conditions (shadow, occlusion, distance)? Cityscapes' known hard classes are the thin, small-instance-count ones — check whether that shows up here or whether something else (lighting, crowding) dominates instead.

## Conclusion & compute budget actual vs. estimate

**Fill in after running**: how did the real per-epoch time compare to the ~6 min/epoch estimate above, and how many compute units did the full run actually cost? If it came in well under or over 25, note what to change (`EPOCHS_FROZEN`/`EPOCHS_FINETUNED`, batch size) for next time.